# Defining Variables
Improves maintainability, it makes migration ​across environments easier, and it avoids repeated hardcoding. 

In [0]:
catalog = "workspace"
schema = "thinknyx"
volume = "thinknyx_vol"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "rows.csv"
table_name = "nyx_table"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema

print(path_volume)
print(path_table)

#Loading CSV File

In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")

#Create a test DataFrame

In [0]:
data = [[2021, "test", "Albany", "M", 42]]
columns = ["Year", "First_Name", "County", "Sex", "Count"]

df1 = spark.createDataFrame(
    data,
    schema="Year int, First_Name STRING, County STRING, Sex STRING, Count int"
)

display(df1)  # The display() method is specific to Databricks notebooks and provides a richer visualization.
# df1.show() The show() method is a part of the Apache Spark DataFrame API and provides basic visualization.

#Load data into a DataFrame from CSV file

In [0]:
df_csv = spark.read.csv(f"{path_volume}/{file_name}",
                        header= True,
                        inferSchema=True,
                        sep=",")
display(df_csv)

#View and interact with your DataFrame

###Print the DataFrame schema

In [0]:
df_csv.printSchema()
df1.printSchema()

In [0]:
# Number of rows
df_csv.count()

In [0]:
# Describe data
df_csv.describe().show()

In [0]:
#Columns names
df_csv.columns

In [0]:
#Display a summary of the dataset
display(df_csv.summary())

#Combining DataFrames

In [0]:
# Union function combines all records of both dataframes into a single dataframe. They have to have the same number of columns and the same column names

df = df1.union(df_csv)
display(df)
df.printSchema

#Filter rows in a DataFrame

In [0]:
# Using filter function

display(df.filter(df["Count"] > 50))

In [0]:
# Using where function

display(df.where(df["Count"] > 50))

#Select columns for a dataframe and order by frequency

In [0]:
from pyspark.sql.functions import desc
display(df.select("First_Name","Count").orderBy(desc("Count")))

#Create a subset of a DataFrame

In [0]:
from pyspark.sql.functions import desc

subsetDF = (
    df.filter(
        (df["Year"] == 2009)
        & (df["Count"] > 100)
        & (df["Sex"] == "F")
    )
    .select("First_Name", "County", "Count")
    .orderBy(desc("Count"))
)

display(subsetDF)

#Rename a column

In [0]:
subsetDF = subsetDF.withColumnRenamed("Count","Total")
display(subsetDF)

#Save the DataFrame into another location

In [0]:
# Save the DataFrame into a table
df.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")

In [0]:
# Save the DataFrame to JSON files

# df.write.format("json").mode("overwrite").save("/tmp/json_data")

df.write.format("json").mode("overwrite").save("/Volumes/workspace/thinknyx/thinknyx_vol/json_data")

In [0]:
display(spark.read.format("json").json("/Volumes/workspace/thinknyx/thinknyx_vol/json_data"))